In [1]:
from pathlib import Path
from datetime import datetime
import re, json
from dotenv import load_dotenv
load_dotenv(Path("configs") / "local.env")

#import numpy as np
#from datasets import load_dataset
import torch
torch.manual_seed(3647)
torch.set_float32_matmul_precision('high')

from transformers import set_seed
set_seed(42)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model_name = "Qwen/Qwen3-8B"

print(f"--> device: {device}")

--> device: cuda


In [3]:
# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,

    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

#quant_config = BitsAndBytesConfig(load_in_8bit=True, bnb_8bit_compute_dtype=torch.bfloat16)
#quant_config = None

In [4]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto",
    quantization_config=quant_config,
)

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [5]:
def chat(messages, max_new_tokens=256, functions=None, temperature=0.2):
    prompt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
        tools=functions,           # 若这里报参名错误，换成 functions=functions
        # functions=functions,
        # enable_thinking=True,    # 若你确认模型是 thinking 版本再开；普通指令版请注释掉
    )

    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,      # 放到 generate 里，而不是 apply_chat_template
        do_sample=(temperature > 0),  # 温度>0时启用采样
        eos_token_id=tokenizer.eos_token_id,
        #enable_thinking=False,
    )

    text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    return text.strip()

def parse_tool_call(text: str):
    """
    兼容两种常见输出：
    1) 纯 JSON：{"name":"get_weather","arguments":{"city":"Shanghai"}}
    2) 带标签：<|tool_call|>{"name":...}</|tool_call|> 或 <tool_call>...</tool_call>
    """

    m = re.search(r"<\|?tool_call\|?>\s*(\{.*?\})\s*</\|?tool_call\|?>", text, flags=re.S)
    if m:
        text = m.group(1)

    text = text.strip()
    # 尝试截断到最后一个 '}'（避免结尾带多余自然语言）
    last_brace = text.rfind("}")
    if last_brace != -1:
        text = text[:last_brace+1]

    try:
        obj = json.loads(text)
        if "name" in obj and "arguments" in obj:
            return obj
    except Exception:
        pass

    return None

In [8]:
functions = [
    {
        "name": "get_weather",
        "description": "Get weather by location",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string"}},
            "required": ["city"],
        },
    },
]

messages = [
    #{"role": "system", "content": "Only return the tool call JSON with keys: name, arguments; no extra text."},
    {"role":"user","content":"What's the weather in Shanghai?"},
]

text = chat(messages, functions=functions)
print(text)

<think>
Okay, the user is asking for the weather in Shanghai. Let me check the tools available. There's a get_weather function that requires a city parameter. Since the user mentioned Shanghai, I need to call that function with the city set to Shanghai. I'll make sure the JSON is correctly formatted with the city name. No other parameters are needed here. Alright, time to generate the tool call.
</think>

<tool_call>
{"name": "get_weather", "arguments": {"city": "Shanghai"}}
</tool_call>


In [9]:
print(parse_tool_call(text))

{'name': 'get_weather', 'arguments': {'city': 'Shanghai'}}
